
## Objective
This notebook focuses on runtime artifact generation for serving in the recommendation system pipeline.

## Data Sources
Primary inputs referenced in this notebook include:
- main_data.csv
- item_catalog.csv
- item_catalog_needs_review.csv
- item_catalog_conflict_details.csv
- category_wise_catalog_items.csv

## Core Workflow
- Import dependencies and initialize configuration.
- Load source datasets into dataframes.
- Aggregate events/items to create behavioral features.
- Persist model/artifacts for reuse.
- Export processed outputs to CSV.

## Expected Outputs
- Processed CSV outputs for downstream stages.

## Role in Pipeline
This notebook contributes notebook-specific assets/signals that feed later candidate generation, ranking, or retraining steps.



<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: import re
**Explanation:** This cell executes logic related to `import re`.

**Possible Output:** Possible output: text/log/value print.


In [1]:
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")
DATA_DIR = BASE_DIR / "data"

ITEM_CATALOG_OUTPUT_DIR = DATA_DIR / "item_catalog_output"
ITEM_CATALOG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_DIR / "main_data.csv"

ITEM_CATALOG_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog.csv"
REVIEW_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog_needs_review.csv"
CONFLICT_FILE = ITEM_CATALOG_OUTPUT_DIR / "item_catalog_conflict_details.csv"

print("Input file exists:", RAW_FILE.exists())
print("Input file:", RAW_FILE)
print("Output folder:", ITEM_CATALOG_OUTPUT_DIR)

Input file exists: True
Input file: C:\D drive\item_recommendation_model_create\data\main_data.csv
Output folder: C:\D drive\item_recommendation_model_create\data\item_catalog_output


<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: df = pd.read_csv(RAW_FILE)
**Explanation:** This cell executes logic related to `df = pd.read_csv(RAW_FILE)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [2]:
df = pd.read_csv(RAW_FILE)

df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Shape: (1000000, 16)
Columns:
['transactionId', 'customerId', 'customerName', 'customerPersona', 'itemId', 'itemName', 'category', 'quantity', 'orderDate', 'isHoliday', 'isFestival', 'season', 'timeSlot', 'dayOfWeek', 'hour', 'month']


,transactionId,customerId,customerName,customerPersona,itemId,itemName,category,quantity,orderDate,isHoliday,isFestival,season,timeSlot,dayOfWeek,hour,month
0,1,23433,MD MOSSIN,family_grocery,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,6,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
1,1,23433,MD MOSSIN,family_grocery,453,Saad Testy Bit Salt-100gm,pantry salt,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
2,1,23433,MD MOSSIN,family_grocery,15262,Cow Brain-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
3,1,23433,MD MOSSIN,family_grocery,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
4,1,23433,MD MOSSIN,family_grocery,13986,Cow Stomach-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: def normalize_text(value):
**Explanation:** This cell executes logic related to `def normalize_text(value):`.

**Possible Output:** Possible output: text/log/value print.


In [3]:
def normalize_text(value):
    if pd.isna(value):
        return value
    
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    
    return value


def normalize_category(category):
    if pd.isna(category):
        return category
    
    category = normalize_text(category)
    
    return category


print("Cleaning functions ready")

Cleaning functions ready


<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: required_cols = ["itemId", "itemName", "category"]
**Explanation:** This cell executes logic related to `required_cols = ["itemId", "itemName", "category"]`.

**Possible Output:** Possible output: text/log/value print.


In [4]:
required_cols = ["itemId", "itemName", "category"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Required columns found")

Required columns found


<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: item_df = df.copy()
**Explanation:** This cell executes logic related to `item_df = df.copy()`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [5]:
item_df = df.copy()

item_df["itemId"] = pd.to_numeric(item_df["itemId"], errors="coerce")
item_df["itemName"] = item_df["itemName"].apply(normalize_text)
item_df["category"] = item_df["category"].apply(normalize_category)

item_df = item_df.dropna(subset=["itemId", "itemName", "category"]).copy()
item_df["itemId"] = item_df["itemId"].astype(int)

print("Shape after cleaning:", item_df.shape)

display(item_df[["itemId", "itemName", "category"]].head())

Shape after cleaning: (1000000, 16)


,itemId,itemName,category
0,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice
1,453,Saad Testy Bit Salt-100gm,pantry salt
2,15262,Cow Brain-K,Meat-Fresh
3,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils
4,13986,Cow Stomach-K,Meat-Fresh


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: print("Unique itemId count:", item_df["itemId"].nunique())
**Explanation:** This cell executes logic related to `print("Unique itemId count:", item_df["itemId"].nunique())`.

**Possible Output:** Possible output: text/log/value print.


In [6]:
print("Unique itemId count:", item_df["itemId"].nunique())
print("Unique itemName count:", item_df["itemName"].nunique())
print("Unique category count:", item_df["category"].nunique())

print("\nTop categories:")
display(item_df["category"].value_counts().head(20))

Unique itemId count: 229
Unique itemName count: 229
Unique category count: 46

Top categories:


category
Pantry-Rice                   82427
Spices-Cooking                58679
Pantry-Oils                   56210
Fish-Fresh                    53323
Meat-Fresh                    52832
Snacks-General                37446
Beverage-Juice                36200
Bakery-Bread                  34297
Beverage-Carbonated           33658
Veg-Cooking                   32634
Protein-Egg                   29152
general_cooking_vegetables    27720
Dairy-Milk                    26242
pantry salt                   25394
Pantry-Flour                  25123
sauce item                    23396
Dairy-Other                   23202
Beverage-Hot                  21798
Pantry-Pulses                 20297
Personal-Care-Bath            19453
Name: count, dtype: int64

<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: item_name_counter = defaultdict(Counter)
**Explanation:** This cell executes logic related to `item_name_counter = defaultdict(Counter)`.

**Possible Output:** Possible output: text/log/value print.


In [7]:
item_name_counter = defaultdict(Counter)
item_category_counter = defaultdict(Counter)
item_price_values = defaultdict(list)
item_quantity_values = defaultdict(list)

for _, row in item_df.iterrows():
    item_id = int(row["itemId"])
    item_name = row["itemName"]
    category = row["category"]
    
    item_name_counter[item_id][item_name] += 1
    item_category_counter[item_id][category] += 1
    
    if "price" in item_df.columns and pd.notna(row.get("price", np.nan)):
        try:
            item_price_values[item_id].append(float(row["price"]))
        except Exception:
            pass
    
    if "quantity" in item_df.columns and pd.notna(row.get("quantity", np.nan)):
        try:
            item_quantity_values[item_id].append(float(row["quantity"]))
        except Exception:
            pass

print("Item history collection completed")

Item history collection completed


<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: catalog_rows = []
**Explanation:** This cell executes logic related to `catalog_rows = []`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [8]:
catalog_rows = []

for item_id in sorted(item_name_counter.keys()):
    canonical_name = item_name_counter[item_id].most_common(1)[0][0]
    canonical_category = item_category_counter[item_id].most_common(1)[0][0]
    
    name_variant_count = len(item_name_counter[item_id])
    category_variant_count = len(item_category_counter[item_id])
    total_rows_seen = sum(item_name_counter[item_id].values())
    
    if len(item_price_values[item_id]) > 0:
        avg_price = round(float(np.mean(item_price_values[item_id])), 2)
        min_price = round(float(np.min(item_price_values[item_id])), 2)
        max_price = round(float(np.max(item_price_values[item_id])), 2)
    else:
        avg_price = np.nan
        min_price = np.nan
        max_price = np.nan
    
    if len(item_quantity_values[item_id]) > 0:
        avg_quantity = round(float(np.mean(item_quantity_values[item_id])), 2)
    else:
        avg_quantity = np.nan
    
    catalog_rows.append({
        "itemId": item_id,
        "itemName": canonical_name,
        "category": canonical_category,
        "avgPrice": avg_price,
        "minPrice": min_price,
        "maxPrice": max_price,
        "avgQuantity": avg_quantity,
        "totalRowsSeen": total_rows_seen,
        "nameVariantCount": name_variant_count,
        "categoryVariantCount": category_variant_count,
        "reviewFlag": 1 if name_variant_count > 1 or category_variant_count > 1 else 0
    })

item_catalog_df = pd.DataFrame(catalog_rows)

print("Item catalog shape:", item_catalog_df.shape)
display(item_catalog_df.head(20))

Item catalog shape: (229, 11)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0
5,133,Rin Advanced-500gm,Household-Laundry,NaN,NaN,NaN,1.41,2085,1,1,0
6,292,Super White Powder-1000g,Household-Laundry,NaN,NaN,NaN,1.41,2266,1,1,0
7,296,Chaka Advanced Ball Soap-125gm,Household-Laundry,NaN,NaN,NaN,1.39,2076,1,1,0
8,318,Musur Pulse Kangaroo -1Kg (sp),Pantry-Pulses,NaN,NaN,NaN,2.67,5192,1,1,0
9,332,ACI Pure Salt-1Kg,pantry salt,NaN,NaN,NaN,1.40,12698,1,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == ...
**Explanation:** This cell executes logic related to `needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == ...`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [9]:
needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == 1].copy()

print("Items needing review:", needs_review_df.shape[0])

display(needs_review_df.head(20))

Items needing review: 0


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag


<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: conflict_rows = []
**Explanation:** This cell executes logic related to `conflict_rows = []`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [10]:
conflict_rows = []

for item_id in sorted(item_name_counter.keys()):
    if len(item_name_counter[item_id]) > 1 or len(item_category_counter[item_id]) > 1:
        conflict_rows.append({
            "itemId": item_id,
            "itemNameVariants": dict(item_name_counter[item_id]),
            "categoryVariants": dict(item_category_counter[item_id])
        })

conflict_details_df = pd.DataFrame(conflict_rows)

print("Conflict details shape:", conflict_details_df.shape)

display(conflict_details_df.head(20))

Conflict details shape: (0, 0)


""


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)
**Explanation:** This cell executes logic related to `item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)`.

**Possible Output:** Possible output: text/log/value print.


In [11]:
item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)
needs_review_df.to_csv(REVIEW_FILE, index=False)
conflict_details_df.to_csv(CONFLICT_FILE, index=False)

print("Saved files:")
print(ITEM_CATALOG_FILE)
print(REVIEW_FILE)
print(CONFLICT_FILE)

Saved files:
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog_needs_review.csv
C:\D drive\item_recommendation_model_create\data\item_catalog_output\item_catalog_conflict_details.csv


<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: final_catalog = pd.read_csv(ITEM_CATALOG_FILE)
**Explanation:** This cell executes logic related to `final_catalog = pd.read_csv(ITEM_CATALOG_FILE)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [12]:
final_catalog = pd.read_csv(ITEM_CATALOG_FILE)

print("Final catalog shape:", final_catalog.shape)
print("Final catalog columns:")
print(final_catalog.columns.tolist())

display(final_catalog.head(20))

Final catalog shape: (229, 11)
Final catalog columns:
['itemId', 'itemName', 'category', 'avgPrice', 'minPrice', 'maxPrice', 'avgQuantity', 'totalRowsSeen', 'nameVariantCount', 'categoryVariantCount', 'reviewFlag']


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0
5,133,Rin Advanced-500gm,Household-Laundry,NaN,NaN,NaN,1.41,2085,1,1,0
6,292,Super White Powder-1000g,Household-Laundry,NaN,NaN,NaN,1.41,2266,1,1,0
7,296,Chaka Advanced Ball Soap-125gm,Household-Laundry,NaN,NaN,NaN,1.39,2076,1,1,0
8,318,Musur Pulse Kangaroo -1Kg (sp),Pantry-Pulses,NaN,NaN,NaN,2.67,5192,1,1,0
9,332,ACI Pure Salt-1Kg,pantry salt,NaN,NaN,NaN,1.40,12698,1,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 13: category_item_check = (
**Explanation:** This cell executes logic related to `category_item_check = (`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [14]:
category_item_check = (
    final_catalog
    .groupby("category")
    .agg(
        totalItems=("itemId", "nunique"),
        itemNames=("itemName", lambda x: list(x))
    )
    .reset_index()
    .sort_values("totalItems", ascending=False)
)

display(category_item_check)

category_item_check.to_csv(
    ITEM_CATALOG_OUTPUT_DIR / "category_wise_catalog_items.csv",
    index=False
)

print("Saved:", ITEM_CATALOG_OUTPUT_DIR / "category_wise_catalog_items.csv")

,category,totalItems,itemNames
32,Snacks-General,19,"[Pran Toast-250g, Sun Chips Mix Masala-35gm, Sun Chips Wasabi-35g, Sun Chips Tomato Tango 38g, Sun Chips Garlic & Ch..."
34,Veg-Cooking,13,"[Tomato L.C(kg), Green Pumpkin-pc, Cucumber Deshi (kg), Green Papaya(kg), Eye Of Arum(Tall) (Kachurmukhi)-kg, Carrot..."
42,household hygene,12,"[Bashu. Toilet Tissue (White), Bashun Paper Towel 1Ply- 250Pcs, B Toilet Tissue Extra Saving Pack 4Rolls, Dettol Ski..."
43,noodles_pasta_and_haleem,11,"[Savory Haleem Mix-200gm, Maggi Saad -E -Magic S Mix-48g, Pran Haleem Mix(200g), EM Pasta Vite-400gm, Knorr Soup-24g..."
7,Dairy-Other,11,"[Rose Water-180ml, Pran Sweetened Yogurt-100gm, Aarong Probiotic Yogurt-500gm, Good Life Mozzarella Cheese-200gm, La..."
33,Spices-Cooking,11,"[Cumin Powder-100gm, Cassia Leaf-100gm (Tespata), Radhuni Spices Cumin-200gm, Radhuni Chatpati Masala 50Gm, Panch Fo..."
10,Fish-Fresh,8,"[Katol Fish (Small)-(MNK), Rui Fish (XL)-(HB), Rui Fish-M-(MNK), Rui Fish (S1)-(MNK), Poa Fish Small-(MNK), Shrimp F..."
11,Fruits-Fresh,8,"[Calender Pineapple, Banana Green Small-kg, Lemon Elachi-Pc, Atta Fruits, Malta (Yellow)-kg, Naspati(kg), Apple (Fuj..."
1,Beverage-Carbonated,8,"[Sprite-1.25 Lt(Pet), Coca Cola-2.25Lt(Pet), Coca Cola-600ml(Pet), Kinley Club Soda -600ml, Mountan Dew-250ml Can, P..."
28,Personal-Care-Oral,7,"[Sensodyne Toothbrush Soft, Sensodyne Deep Clean Toothbrush, Ambition Tooth brush, Pepsodent Gremichek+ Toothpaste-1..."


Saved: C:\D drive\item_recommendation_model_create\data\item_catalog_output\category_wise_catalog_items.csv
